# Stage 2 — Project + analyze

A100 GPU. Upload `value_axis.npy` (or dev: `value_axis_proxy.npy`) and `normalized.zip` from ingest.

In [ ]:
PRESET = 'dev'  # 'default' | 'dev'
import torch
assert torch.cuda.is_available()

In [ ]:
!git clone https://github.com/abdelmagid07/latent_failiure_prediction.git
%cd latent_failiure_prediction
!pip install -q -e stage1 -e stage2

In [ ]:
import os, shutil, glob, zipfile
from google.colab import files

os.makedirs('stage1/data', exist_ok=True)
for name in files.upload():
    shutil.move(name, f'stage1/data/{name}')

os.makedirs('stage2/data/normalized', exist_ok=True)
zname = next(iter(files.upload()))
with zipfile.ZipFile(zname) as z:
    z.extractall('stage2/data/normalized_extracted')
for p in glob.glob('stage2/data/normalized_extracted/**/*.json', recursive=True):
    shutil.move(p, f'stage2/data/normalized/{os.path.basename(p)}')

In [ ]:
axis = '../stage1/data/value_axis_proxy.npy' if PRESET == 'dev' else '../stage1/data/value_axis.npy'
%cd stage2
!python -m stage2.extract.project_steps --traj-dir data/normalized \
  --output data/projections.parquet --axis-path {axis}

In [ ]:
!python -m stage2.analyze.run_analyses \
  --projections data/projections.parquet \
  --output-dir data/analysis_report

In [ ]:
# Download cell 
import zipfile, os
from google.colab import files

out = 'data/analysis_report'
targets = ['analysis_report.json',        
           'final_step_separation.png',
           'noise_by_token_type.png',
           'snr_by_position.csv']

with zipfile.ZipFile('stage2_results.zip', 'w') as z:
    for name in targets:
        p = os.path.join(out, name)
        assert os.path.exists(p), f'MISSING: {p}'  
        z.write(p, name)

files.download('stage2_results.zip')